# Social Media Virality Prediction — Final Model Notebook

This notebook:
- loads the 150,000-row social-media dataset,
- creates 4 engineered features (31 → 35 columns),
- creates a leakage-safe binary virality target from the top 25% of `viral_coefficient`,
- compares **Logistic Regression** and **CatBoost**,
- tunes the classification threshold on validation data,
- evaluates once on a held-out test set,
- retrains the selected CatBoost model on train + validation data,
- saves the final reusable model bundle as **`social_media_virality_best_model.pkl`**.

Engagement-derived engineered features are kept in the dataset but intentionally excluded from model inputs because they contain post-performance information.

In [ ]:
# Install CatBoost only if it is missing.
try:
    import catboost
except ImportError:
    %pip install -q catboost


In [ ]:
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve
)
from catboost import CatBoostClassifier

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)


## 1. Load the dataset

In [ ]:
DATASET_NAME = 'multi_platform_social_sentiment_evolution.csv'

candidates = [
    Path(DATASET_NAME),
    Path('/content') / DATASET_NAME,
    Path('/mnt/data') / DATASET_NAME,
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        f'Could not find {DATASET_NAME}. Upload/place the CSV beside this notebook and run again.'
    )

df = pd.read_csv(data_path)
print('Dataset:', data_path)
print('Original shape:', df.shape)
display(df.head())


In [ ]:
# Basic integrity checks
print('Duplicate full rows:', df.duplicated().sum())
print('Missing values:', int(df.isna().sum().sum()))
print('Unique post_id:', df['post_id'].nunique(), '/', len(df))
print('Date range:', df['date'].min(), 'to', df['date'].max())


## 2. Feature engineering: 31 → 35 columns

In [ ]:
# Four engineered features requested for the dataset.
# +1 in denominators avoids division by zero and extreme instability for zero-valued fields.
df['engagement_per_view'] = df['total_engagement'] / (df['views'] + 1)
df['engagement_velocity'] = df['total_engagement'] / (df['hours_since_post'] + 1)
df['follower_growth_rate'] = df['followers'] / (df['account_age_days'] + 1)
df['discussion_ratio'] = df['comments'] / (df['total_engagement'] + 1)

print('Shape after feature engineering:', df.shape)
assert df.shape[1] == 35, f'Expected 35 columns after feature engineering, got {df.shape[1]}'
display(df[['engagement_per_view','engagement_velocity','follower_growth_rate','discussion_ratio']].describe())


## 3. Create the virality target

A post is labeled **viral (1)** when its `viral_coefficient` is above the dataset's 75th percentile. This produces a useful 75/25 binary classification problem rather than an extremely rare-event target.


In [ ]:
VIRAL_QUANTILE = 0.75
viral_coefficient_threshold = float(df['viral_coefficient'].quantile(VIRAL_QUANTILE))
df['is_viral'] = (df['viral_coefficient'] > viral_coefficient_threshold).astype(int)

print(f'75th percentile viral_coefficient threshold: {viral_coefficient_threshold:.4f}')
print()
print('Target counts:')
print(df['is_viral'].value_counts().sort_index())
print()
print('Target percentages:')
print((df['is_viral'].value_counts(normalize=True).sort_index() * 100).round(2))


## 4. Select leakage-safe predictors

The model is intended to predict virality from information about the post/account rather than from the post's eventual success. Therefore the following are **not used as predictors**: `likes`, `shares`, `comments`, `views`, `total_engagement`, `engagement_rate_per_1k_followers`, `engagement_per_view`, `engagement_velocity`, `discussion_ratio`, `viral_coefficient`, and `cross_platform_spread`.

`follower_growth_rate` is safe because it depends only on account age and follower count.

In [ ]:
FEATURES = [
    'platform',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'followers', 'account_age_days', 'verified',
    'topic', 'language', 'content_length', 'media_type', 'num_hashtags',
    'sentiment_category', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral',
    'toxicity_score', 'location',
    'follower_growth_rate'
]

CATEGORICAL_FEATURES = [
    'platform', 'day_of_week', 'topic', 'language',
    'media_type', 'sentiment_category', 'location'
]
NUMERIC_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

X = df[FEATURES].copy()
y = df['is_viral'].copy()

print('Model input features:', len(FEATURES))
print('Categorical:', len(CATEGORICAL_FEATURES))
print('Numeric/binary:', len(NUMERIC_FEATURES))


## 5. Train / validation / test split

In [ ]:
# 70% training, 15% validation, 15% final test; stratification preserves class proportions.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', X_train.shape, y_train.mean().round(4))
print('Validation:', X_val.shape, y_val.mean().round(4))
print('Test:', X_test.shape, y_test.mean().round(4))


## 6. Helper functions

In [ ]:
def best_f1_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx]), float(f1[idx])


def evaluate_model(name, model, X_eval, y_eval, threshold):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_eval, pred),
        'Precision': precision_score(y_eval, pred, zero_division=0),
        'Recall': recall_score(y_eval, pred, zero_division=0),
        'F1': f1_score(y_eval, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, proba),
        'PR-AUC': average_precision_score(y_eval, proba),
        'Threshold': threshold,
        'Predictions': pred,
        'Probabilities': proba,
    }


## 7. Model 1 — Logistic Regression baseline

In [ ]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
        ('numeric', StandardScaler(), NUMERIC_FEATURES),
    ]
)

logistic_model = Pipeline([
    ('preprocess', logistic_preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)
logistic_val_proba = logistic_model.predict_proba(X_val)[:, 1]
logistic_threshold, logistic_val_f1 = best_f1_threshold(y_val, logistic_val_proba)
print(f'Best Logistic Regression validation threshold: {logistic_threshold:.4f}')
print(f'Validation F1 at threshold: {logistic_val_f1:.4f}')


## 8. Model 2 — CatBoost

In [ ]:
# CatBoost handles categorical features directly.
def prepare_catboost_frame(frame):
    out = frame.copy()
    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].astype(str)
    return out

X_train_cb = prepare_catboost_frame(X_train)
X_val_cb = prepare_catboost_frame(X_val)
X_test_cb = prepare_catboost_frame(X_test)

catboost_model = CatBoostClassifier(
    iterations=350,
    depth=7,
    learning_rate=0.06,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False
)

catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=50,
    verbose=False
)

catboost_val_proba = catboost_model.predict_proba(X_val_cb)[:, 1]
catboost_threshold, catboost_val_f1 = best_f1_threshold(y_val, catboost_val_proba)

print('Best CatBoost iteration:', catboost_model.get_best_iteration() + 1)
print(f'Best CatBoost validation threshold: {catboost_threshold:.4f}')
print(f'Validation F1 at threshold: {catboost_val_f1:.4f}')


## 9. Final held-out test comparison

In [ ]:
lr_result = evaluate_model(
    'Logistic Regression', logistic_model, X_test, y_test, logistic_threshold
)
cb_result = evaluate_model(
    'CatBoost', catboost_model, X_test_cb, y_test, catboost_threshold
)

comparison = pd.DataFrame([
    {k: v for k, v in lr_result.items() if k not in ['Predictions','Probabilities']},
    {k: v for k, v in cb_result.items() if k not in ['Predictions','Probabilities']},
])

metric_cols = ['Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC','Threshold']
comparison[metric_cols] = comparison[metric_cols].round(4)
display(comparison)


In [ ]:
print('CatBoost classification report on the held-out test set:')
print(classification_report(y_test, cb_result['Predictions'], digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_test, cb_result['Predictions']))


## 10. CatBoost feature importance

In [ ]:
importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': catboost_model.get_feature_importance()
}).sort_values('importance', ascending=False).reset_index(drop=True)

display(importance)

plt.figure(figsize=(9, 6))
plt.barh(importance.head(12)['feature'][::-1], importance.head(12)['importance'][::-1])
plt.xlabel('CatBoost feature importance')
plt.title('Top 12 Virality Predictors')
plt.tight_layout()
plt.show()


## 11. Retrain the selected final model and save `.pkl`

CatBoost is selected using the validation/test comparison, especially **F1, ROC-AUC and PR-AUC**. It is retrained on the combined training + validation data using the number of boosting iterations selected by early stopping. The held-out test set is not included in final training.

In [ ]:
best_iterations = int(catboost_model.get_best_iteration() + 1)

X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)
X_trainval_cb = prepare_catboost_frame(X_trainval)

final_model = CatBoostClassifier(
    iterations=best_iterations,
    depth=7,
    learning_rate=0.06,
    loss_function='Logloss',
    auto_class_weights='Balanced',
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False
)

final_model.fit(
    X_trainval_cb,
    y_trainval,
    cat_features=CATEGORICAL_FEATURES,
    verbose=False
)

# Recheck performance on the untouched test set after retraining.
final_test_result = evaluate_model(
    'Final CatBoost', final_model, X_test_cb, y_test, catboost_threshold
)

print('Final CatBoost held-out test metrics:')
for key in ['Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']:
    print(f'{key}: {final_test_result[key]:.4f}')


In [ ]:
# Save a complete bundle rather than only the estimator.
# This preserves everything needed to reproduce the exact prediction setup.
model_bundle = {
    'model': final_model,
    'model_name': 'CatBoostClassifier',
    'feature_columns': FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'decision_threshold': float(catboost_threshold),
    'viral_coefficient_threshold': float(viral_coefficient_threshold),
    'viral_target_quantile': VIRAL_QUANTILE,
    'engineered_feature_formula': {
        'follower_growth_rate': 'followers / (account_age_days + 1)'
    },
    'random_state': RANDOM_STATE,
    'test_metrics': {
        key: float(final_test_result[key])
        for key in ['Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']
    }
}

OUTPUT_PKL = Path('social_media_virality_best_model.pkl')
with open(OUTPUT_PKL, 'wb') as f:
    pickle.dump(model_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved final model bundle: {OUTPUT_PKL.resolve()}')
print(f'File size: {OUTPUT_PKL.stat().st_size / (1024**2):.2f} MB')


## 12. Verify that the saved `.pkl` loads and predicts

In [ ]:
with open(OUTPUT_PKL, 'rb') as f:
    loaded_bundle = pickle.load(f)

sample = X_test.head(5).copy()
sample['follower_growth_rate'] = sample['followers'] / (sample['account_age_days'] + 1)
for col in loaded_bundle['categorical_features']:
    sample[col] = sample[col].astype(str)

sample_prob = loaded_bundle['model'].predict_proba(
    sample[loaded_bundle['feature_columns']]
)[:, 1]
sample_pred = (sample_prob >= loaded_bundle['decision_threshold']).astype(int)

verification = pd.DataFrame({
    'viral_probability': sample_prob.round(4),
    'predicted_is_viral': sample_pred
})
display(verification)
print('PKL verification successful.')
